# Fitting the BlueJay photometry with Bagpipes
In analogy to Prospector

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np 
import matplotlib as mpl
mpl.rcParams["text.usetex"] = True

from astropy.cosmology import WMAP9 as cosmo
import bagpipes as pipes
import matplotlib.pyplot as plt
#%matplotlib inline

from astropy.io import fits
from astropy.table import Table

import mockutils as mock

# Replicating the Prospector Model

Specify all parameters such that they match the Prospector model that Letizia used for my galaxies.

Prospector Parameter    Bagpipes Equivalent     Translation Notes
zred                    redshift                "Use (1.764, 0.5) for a Gaussian."
logzsol                 metallicity             Convert log to linear (10−2 to 100.5).
logmass                 massformed              "Bagpipes uses total formed mass, not current stellar mass."
dust2                   Av                      AV​=1.086×optical depth.
dust_index              eta                     Bagpipes' cf00 model uses eta for the slope.

Set up the `fit_instructions` dictionary

In [ ]:
fit_instructions = {}                       # The fit instructions dictionary

# Nebular component
nebular = {}
nebular["logU"] = (-4., -1.)                # Log_10 of the ionisation parameter.
#nebular["fesc"] = (0., 1.)                  # IMPORTANT: Escape fraction of ionising photons. Standard value is 0.1

# Dust absorption parameters
dust = {}                               # Dust component
dust["type"] = "CF00"                   # Define the shape of the attenuation curve
dust["Av"] = (0., 4.0)                  # Vary Av between 0 and 3.2 magnitudes
dust["n"] = (-1.0, 1.5)                 # Vary the slope of the attenuation curve from -1.0 to 1.5
dust["n_prior"] = "Gaussian"            # Set a Gaussian prior
dust["n_prior_mu"] = 0.7                # Centred on the standard CF00 slope of 0.7
dust["n_prior_sigma"] = 0.3             # With a width of 0.3

# Dust emission parameters (now free parameters)
dust["qpah"] = (0.1, 4.58)                  # PAH mass fraction (spanning the entire grid)
dust["umin"] = (0.1, 25.)                   # Lower limit of starlight intensity distribution (spanning the entire grid)
dust["gamma"] = (0., 1.0)                   # Fraction of stars at Umin

# Alternatively use a very narrow Gaussian prior centred on the spectroscopic redshift:
#fit_instructions["redshift"] = (zred-1, zred+1)     # Set the redshift prior to vary within +/1 of the spectroscopic redshift
#fit_instructions["redshift_prior"] = "Gaussian"     # Use a Gaussian prior
#fit_instructions["redshift_prior_mu"] = zred        # Centred on the spectroscopic redshift
#fit_instructions["redshift_prior_sigma"] = 0.005    # With a very narrow width

# Setting the SFH priors
continuity = {}
continuity["massformed"] = (8.5, 13)            # Log10 of solar mass formed
continuity["metallicity"] = (0.01, 3.16)        # Linear Z/Z_solar: 0.01 to 3.16
continuity["metallicity_prior"] = "log_10"      # Logarithmic prior for metallicity

# Define the dsfr ratios
for i in range(1, 7):   # for 7 bins
    continuity[f"dsfr{i}"] = (-10., 10.) 
    continuity[f"dsfr{i}_prior"] = "student_t"
    #continuity["dsfr" + str(i) + "_prior_df"] = 2       # Default nu value of 2 in the student_t distribution
    #continuity["dsfr" + str(i) + "_prior_scale"] = 0.3  # Default sigma value of 0.3 in the student_t distribution

fit_instructions["dust"] = dust
fit_instructions["nebular"] = nebular
fit_instructions["continuity"] = continuity

print(fit_instructions)

Now run the galaxy with minimal parameters

In [ ]:
run = "bluejay_with_alma"

# Step 1: Get all IDs from the MIRI catalogue
#miri_table = "data/catalogues/Phot_Table_MIRI.fits"
#id_table = Table.read(miri_table)
#all_ids = [int(val.decode('utf-8') if isinstance(val, bytes) else val) 
#                        for val in id_table['ID']]

ALMA = {"filters/bluejay_alma6.txt": [8280],
        "filters/bluejay_alma7.txt": [13103, 18252, 21165]}  # ALMA bands in microns

# Step 2: Loop over all IDs, load the galaxy, set the redshift and SFH priors, and fit each one!
for filt_file, gal_ids in ALMA.items():
    
    filt_list = np.loadtxt(filt_file, dtype="str")
    
    for objid in gal_ids:
        posterior_dir = f'/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/pipes/posterior/{run}'
        
        filename = os.path.join(posterior_dir, f"{objid}.h5")
            
        if os.path.exists(filename):
            print(f"Skipping galaxy {objid} - output file already exists!")
            continue
        
        # Create galaxy object for the current ID
        galaxy = pipes.galaxy(f"{objid}", mock.load_bluejay_with_alma, spectrum_exists=False, filt_list=filt_list)

        # Load the redshift
        zred, is_spectroscopic = mock.get_zred(objid)
        if zred is None:
            print(f"No valid redshift found for ID {objid}. Skipping this galaxy.")
            continue

        if is_spectroscopic:
            print(f"Fixing redshift to the spectroscopic value of the source: {zred}")
            fit_instructions["redshift"] = zred                       # Fix the redshift to the spectroscopic value to reduce runtime
        else:
            print(f"Using photometric redshift of {zred} with a Gaussian prior for the source.")
            fit_instructions["redshift"] = (zred-0.25, zred+0.25)     # Set the redshift prior to vary within +/1 of the photometric redshift
            fit_instructions["redshift_prior"] = "Gaussian"           # Use a Gaussian prior
            fit_instructions["redshift_prior_mu"] = zred              # Centred on the photometric redshift
            fit_instructions["redshift_prior_sigma"] = 0.05           # With a width of 0.05

        # Calculate the agebins for the star formation history of the galaxy based on its redshift and update the fit instructions
        age_bins = mock.zred_to_agebins(zred, nbins_sfh=7)
        print("Age bins:", age_bins)
        fit_instructions["continuity"]["bin_edges"] = age_bins   
        
        fit = pipes.fit(galaxy, fit_instructions, run=run)

        fit.fit(verbose=True, sampler='nautilus', pool=6, discard_exploration=True)

        try:
            fig = fit.plot_spectrum_posterior(save=True, show=False)
        except ValueError:
            print(f"Warning: Spectrum plot failed for {objid} due to empty photometry array.")

        try:
            fig = fit.plot_sfh_posterior(save=True, show=False)
            fig = fit.plot_corner(save=True, show=False)
        except Exception as e:
            print(f"Warning: Other plots failed for {objid}: {e}")

Load example posterior

In [ ]:
#list(fit.posterior.samples['sfr'])
list(fit.posterior.samples['nebular:fesc'])

posterior = fit.posterior.samples

print(np.shape(fit.posterior.samples['nebular:fesc']))
print(np.shape(fit.posterior.samples['sfr']))

fesc_array = fit.posterior.samples['nebular:fesc']

print(np.median(fesc_array))

# Fit the data without MIRI

In [ ]:
import numpy as np 
import matplotlib as mpl
mpl.rcParams["text.usetex"] = True

from astropy.cosmology import WMAP9 as cosmo
import bagpipes as pipes
import matplotlib.pyplot as plt
#%matplotlib inline

from astropy.io import fits
from astropy.table import Table

def load_bluejay(ID):
    """ Load BlueJay photometry from the BlueJay catalogue(s)"""

    # Blue Jay catalogue
    bluejay_cat = Table.read("bluejay_phot_cat_v1.4.fits")
    
    # 1. List all available HST/NIRCam bands:
    filters = ['F090W', 'F115W', 'F125W', 'F140W', 'F150W', 'F160W', 
               'F200W', 'F277W', 'F356W', 'F410M', 'F444W', 'F606W', 'F814W']
    
    # 2. Find the correct row using the ID column
    # Use a mask rather than (int(ID) - 1) to be safe against non-sequential IDs
    row = bluejay_cat[bluejay_cat['ID'] == int(ID)]

    if len(row) == 0:
        raise ValueError(f"ID {ID} not found in catalogue.")
    
    # 3. Extract fluxes and errors into lists
    fluxes = []
    flux_errs = []

    for f in filters:
        fluxes.append(row[f + "_flux"][0] * 1e6)
        flux_errs.append(row[f + "_flux_err"][0] * 1e6)
    
    # Now turn these into a 2D array [N_filters, 2]
    # Bagpipes expects photometry[i, 0] = flux, photometry[i, 1] = error
    photometry = np.c_[fluxes, flux_errs]
    
    # 5. Clean up missing data and enforce SNR limits
    for i in range(len(photometry)):
        # Blow up errors for missing data (NaN or 0 flux)
        if (photometry[i, 0] <= 0.) or (np.isnan(photometry[i, 0])):
            photometry[i, :] = [0., 9.9e99]
            continue # Skip SNR check for bad data
            
    return photometry

bluejay_filt_list = np.loadtxt("filters/bluejay_filt_list_nomiri.txt", dtype="str")

galaxy = pipes.galaxy("12020", load_bluejay, spectrum_exists=False, filt_list=bluejay_filt_list)

fig = galaxy.plot()

In [ ]:
# Redshift
def get_zred(galaxy_id):    
    # --- Read Blue Jay catalogue ---
    blue = "/Users/benjamincollins/University/Master/BlueJay/BlueJay_sample.txt"
    tbl = Table.read(blue, format="ascii.basic")
    
    row = tbl[tbl['id'] == int(galaxy_id)]
    
    # Make sure that the code doesn't crash if it can't find the ID in the catalogue
    if len(row) == 0:   
        return None, False
    
    z_spec = row['z_spec'][0]
    
    if z_spec is not None and not np.isnan(z_spec):
        return z_spec, True
    else:
        z_phot = row['z_phot'][0]
        return z_phot, False

# Star formation histories
def zred_to_agebins(zred, z_limit_sfh=20.0, nbins_sfh=8):
    tuniv = cosmo.age(zred).value*1e9   # Age of the universe at the observed redshift in years
    #tbinmax = tuniv-cosmo.age(z_limit_sfh).value*1e9 # Maximum age bin edge corresponding to z_limit_sfh
    tbinmax = tuniv*0.95
    # Compute edges in logarithmic space
    log_edges = np.append(np.array([0.0, 6.7, 7.0]), np.linspace(7.0, np.log10(tbinmax), int(nbins_sfh-1))[1:])
    bin_edges = 10**log_edges   # Convert back to linear space
    bin_edges /= 1e6 # ensure that edges are in Myr for Bagpipes
    return bin_edges.tolist()   # return list of age bin edges

fit_instructions = {}                       # The fit instructions dictionary

# Nebular component
nebular = {}
nebular["logU"] = (-4., -1.)                # Log_10 of the ionisation parameter.
nebular["fesc"] = (0., 1.)                  # IMPORTANT: Escape fraction of ionising photons. Standard value is 0.1

# Dust absorption parameters
dust = {}                               # Dust component
dust["type"] = "CF00"                   # Define the shape of the attenuation curve
dust["Av"] = (0., 4.0)                  # Vary Av between 0 and 3.2 magnitudes
dust["n"] = (-1.0, 1.5)                 # Vary the slope of the attenuation curve from -1.0 to 1.5
dust["n_prior"] = "Gaussian"            # Set a Gaussian prior
dust["n_prior_mu"] = 0.7                # Centred on the standard CF00 slope of 0.7
dust["n_prior_sigma"] = 0.3             # With a width of 0.3

# Dust emission parameters (now free parameters)
dust["qpah"] = (0.1, 4.58)                  # PAH mass fraction (spanning the entire grid)
dust["umin"] = (0.1, 25.)                   # Lower limit of starlight intensity distribution (spanning the entire grid)
dust["gamma"] = (0., 1.0)                   # Fraction of stars at Umin

# Alternatively use a very narrow Gaussian prior centred on the spectroscopic redshift:
#fit_instructions["redshift"] = (zred-1, zred+1)     # Set the redshift prior to vary within +/1 of the spectroscopic redshift
#fit_instructions["redshift_prior"] = "Gaussian"     # Use a Gaussian prior
#fit_instructions["redshift_prior_mu"] = zred        # Centred on the spectroscopic redshift
#fit_instructions["redshift_prior_sigma"] = 0.005    # With a very narrow width

# Setting the SFH priors
continuity = {}
continuity["massformed"] = (8.5, 13)            # Log10 of solar mass formed
continuity["metallicity"] = (0.01, 3.16)        # Linear Z/Z_solar: 0.01 to 3.16
continuity["metallicity_prior"] = "log_10"      # Logarithmic prior for metallicity

# Define the dsfr ratios
for i in range(1, 7):   # for 7 bins
    continuity[f"dsfr{i}"] = (-10., 10.) 
    continuity[f"dsfr{i}_prior"] = "student_t"
    #continuity["dsfr" + str(i) + "_prior_df"] = 2       # Default nu value of 2 in the student_t distribution
    #continuity["dsfr" + str(i) + "_prior_scale"] = 0.3  # Default sigma value of 0.3 in the student_t distribution

fit_instructions["dust"] = dust
fit_instructions["nebular"] = nebular
fit_instructions["continuity"] = continuity

print(fit_instructions)

In [ ]:
run = "fesc_with_miri"

# Step 1: Get all IDs from the MIRI catalogue
miri_table = "./Phot_Table_MIRI.fits"
id_table = Table.read(miri_table)
all_ids = [int(val.decode('utf-8') if isinstance(val, bytes) else val) 
                        for val in id_table['ID']]

# Step 2: Loop over all IDs, load the galaxy, set the redshift and SFH priors, and fit each one!
for objid in all_ids:
    
    posterior_dir = f'/Users/benjamincollins/University/PhD/Code/bagpipes/examples/pipes/posterior/{run}'
    
    filename = os.path.join(posterior_dir, f"{objid}.h5")
        
    if os.path.exists(filename):
        print(f"Skipping galaxy {objid} - output file already exists!")
        continue
    
    # Create galaxy object for the current ID
    galaxy = pipes.galaxy(f"{objid}", load_bluejay, spectrum_exists=False, filt_list=bluejay_filt_list)

    # Load the redshift
    zred, is_spectroscopic = get_zred(objid)
    if zred is None:
        print(f"No valid redshift found for ID {objid}. Skipping this galaxy.")
        continue

    if is_spectroscopic:
        print(f"Fixing redshift to the spectroscopic value of the source: {zred}")
        fit_instructions["redshift"] = zred                       # Fix the redshift to the spectroscopic value to reduce runtime
    else:
        print(f"Using photometric redshift of {zred} with a Gaussian prior for the source.")
        fit_instructions["redshift"] = (zred-0.25, zred+0.25)     # Set the redshift prior to vary within +/1 of the photometric redshift
        fit_instructions["redshift_prior"] = "Gaussian"           # Use a Gaussian prior
        fit_instructions["redshift_prior_mu"] = zred              # Centred on the photometric redshift
        fit_instructions["redshift_prior_sigma"] = 0.05           # With a width of 0.05

    # Calculate the agebins for the star formation history of the galaxy based on its redshift and update the fit instructions
    age_bins = zred_to_agebins(zred, nbins_sfh=7)
    print("Age bins:", age_bins)
    fit_instructions["continuity"]["bin_edges"] = age_bins   
    
    fit = pipes.fit(galaxy, fit_instructions, run=run)

    fit.fit(verbose=True, sampler='nautilus', pool=6, discard_exploration=True)

    fig = fit.plot_spectrum_posterior(save=True, show=False)
    fig = fit.plot_sfh_posterior(save=True, show=False)
    fig = fit.plot_corner(save=True, show=False) 

# Fit the mock galaxy with Bagpipes

In [ ]:
import os
import numpy as np 
import matplotlib as mpl
mpl.rcParams["text.usetex"] = True

from astropy.cosmology import WMAP9 as cosmo
import bagpipes as pipes
import matplotlib.pyplot as plt
#%matplotlib inline

from astropy.io import fits
from astropy.table import Table

import mockutils as mock

fit_instructions = {}                       # The fit instructions dictionary

# Nebular component
nebular = {}
nebular["logU"] = (-4., -1.)                # Log_10 of the ionisation parameter.
#nebular["fesc"] = (0., 1.)                  # IMPORTANT: Escape fraction of ionising photons. Standard value is 0.1

# Dust absorption parameters
dust = {}                               # Dust component
dust["type"] = "CF00"                   # Define the shape of the attenuation curve
dust["Av"] = (0., 4.0)                  # Vary Av between 0 and 3.2 magnitudes
dust["n"] = (-1.0, 1.5)                 # Vary the slope of the attenuation curve from -1.0 to 1.5
dust["n_prior"] = "Gaussian"            # Set a Gaussian prior
dust["n_prior_mu"] = 0.7                # Centred on the standard CF00 slope of 0.7
dust["n_prior_sigma"] = 0.3             # With a width of 0.3

# Dust emission parameters (now free parameters)
dust["qpah"] = (0.1, 4.58)                  # PAH mass fraction (spanning the entire grid)
dust["umin"] = (0.1, 25.)                   # Lower limit of starlight intensity distribution (spanning the entire grid)
dust["gamma"] = (0., 1.0)                   # Fraction of stars at Umin

# Alternatively use a very narrow Gaussian prior centred on the spectroscopic redshift:
zred = 1.00
fit_instructions["redshift"] = (zred-1, zred+1)     # Set the redshift prior to vary within +/1 of the spectroscopic redshift
fit_instructions["redshift_prior"] = "Gaussian"     # Use a Gaussian prior
fit_instructions["redshift_prior_mu"] = zred        # Centred on the spectroscopic redshift
fit_instructions["redshift_prior_sigma"] = 0.005    # With a very narrow width

# Setting the SFH priors
continuity = {}
continuity["massformed"] = (8.5, 13)            # Log10 of solar mass formed
continuity["metallicity"] = (0.01, 3.16)        # Linear Z/Z_solar: 0.01 to 3.16
continuity["metallicity_prior"] = "log_10"      # Logarithmic prior for metallicity

# Define the dsfr ratios
for i in range(1, 7):   # for 7 bins
    continuity[f"dsfr{i}"] = (-10., 10.) 
    continuity[f"dsfr{i}_prior"] = "student_t"
    #continuity["dsfr" + str(i) + "_prior_df"] = 2       # Default nu value of 2 in the student_t distribution
    #continuity["dsfr" + str(i) + "_prior_scale"] = 0.3  # Default sigma value of 0.3 in the student_t distribution

fit_instructions["dust"] = dust
fit_instructions["nebular"] = nebular
fit_instructions["continuity"] = continuity

print(fit_instructions)

Don't forget to specify the correct redshift above before you run the fits!!

In [ ]:
mock_id = 9996

custom_filt_list = np.loadtxt("filters/hst_wfc_nircam_w_miri_all.txt", dtype="str")

my_custom_loader = mock.make_loader(filtlist=custom_filt_list, fit_true_phot=False)

run = "hst_wfc_nircam_w_miri_all"

# Create galaxy object for the current ID
galaxy = pipes.galaxy(mock_id, 
                      load_data=my_custom_loader, 
                      spectrum_exists=False, 
                      filt_list=custom_filt_list)

# Calculate the agebins for the star formation history of the galaxy based on its redshift and update the fit instructions
age_bins = mock.zred_to_agebins(zred, nbins_sfh=7)
print("Age bins:", age_bins)
fit_instructions["continuity"]["bin_edges"] = age_bins   

fit = pipes.fit(galaxy, fit_instructions, run=run)

fit.fit(verbose=True, n_live=800, sampler='nautilus', pool=6, discard_exploration=True)

fig = fit.plot_spectrum_posterior(save=True, show=False)
fig = fit.plot_sfh_posterior(save=True, show=False)
fig = fit.plot_corner(save=True, show=False) 